# taller-evaluativo-03-numpy
## Nombre: Juan Camilo Blanco Parra


### Ejercicio 1 — Crear los arreglos y leer su etiqueta
Se importa NumPy con el alias `np`. Se crea el arreglo bidimensional `lecturas_crudas` a partir de una lista de listas con los datos originales de los cinco sensores, extrayendo dinámicamente sus atributos `shape`, `ndim`, `size` y `dtype`. Asimismo, se generan los días con `np.arange` y los arreglos 1D de pendientes e interceptos.

In [1]:
import numpy as np

# Creación de lecturas_crudas (5 filas x 7 columnas)
lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0,  9.5],   # SBC-A
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],   # SBC-B
    [ 8.0, 11.0, 19.0, 27.0, 33.0, 21.0,  6.0],   # SBC-C
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],   # SBC-D
    [ 5.0,  9.0, 16.0, 22.0, 45.0, 30.0,  4.0],   # SBC-E
])

# Lectura dinámica de atributos
forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

# Días del 1 al 7
dias = np.arange(1, 8)

# Parámetros de calibración (SBC-A a SBC-E)
pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print("Forma:", forma)
print("Dimensiones:", dimensiones)
print("Total elementos:", total)
print("Tipo de dato:", tipo)
print("Días:", dias)
print("Pendientes:", pendientes)
print("Interceptos:", interceptos)

Forma: (5, 7)
Dimensiones: 2
Total elementos: 35
Tipo de dato: float64
Días: [1 2 3 4 5 6 7]
Pendientes: [1.2 0.9 1.1 1.  1.3]
Interceptos: [-0.5  1.  -2.   0.5 -1.5]


### Ejercicio 2 — Señalar celdas y recortar bloques
Se utiliza indexación y slicing avanzados de NumPy sobre `lecturas_crudas` para extraer valores específicos, filas completas, columnas y bloques rectangulares con una sola expresión por cada requerimiento, sin duplicar datos a mano.

In [2]:
lectura_c_dia4 = lecturas_crudas[2, 3]       # SBC-C (fila 2), día 4 (columna 3)
ultimo_primer = lecturas_crudas[-1, 0]            # Último sensor (SBC-E), día 1
dia5 = lecturas_crudas[:, 4]                        # Columna completa del día 5
semana_b = lecturas_crudas[1, :]                        # Fila completa de SBC-B
bloque = lecturas_crudas[1:4, 2:6]                 # Filas 1 a 3, columnas 2 a 5

print("lectura_c_dia4:", lectura_c_dia4)
print("ultimo_primer:", ultimo_primer)
print("dia5:", dia5)
print("semana_b:", semana_b)
print("bloque:\n", bloque)
print("Forma del bloque:", bloque.shape)

lectura_c_dia4: 27.0
ultimo_primer: 5.0
dia5: [28. 41. 33. 52. 45.]
semana_b: [12. 20. 35. 48. 41. 25. 14.]
bloque:
 [[35. 48. 41. 25.]
 [19. 27. 33. 21.]
 [31. 44. 52. 36.]]
Forma del bloque: (3, 4)


### Ejercicio 3 — Limpiar una campaña con máscaras booleanas
Se define la serie de prueba de SBC-A y se construyen máscaras booleanas combinadas con `&` y entre paréntesis para detectar el centinela `-999.0` y aislar las lecturas útiles dentro del rango físico permitido `[0, 500.0]`.

In [3]:
campana_sbc_a = np.array([10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0])

faltantes = (campana_sbc_a == -999.0)
es_util = (campana_sbc_a >= 0.0) & (campana_sbc_a <= 500.0)

lecturas_utiles = campana_sbc_a[es_util]
promedio_utiles = lecturas_utiles.mean()

print("Cantidad de lecturas faltantes:", int(faltantes.sum()))
print("Lecturas útiles:", lecturas_utiles)
print("Promedio de útiles:", promedio_utiles)

Cantidad de lecturas faltantes: 2
Lecturas útiles: [10. 25. 40.  8. 52.]
Promedio de útiles: 27.0


### Ejercicio 4 — Calibrar todo con operaciones vectorizadas y broadcasting
*Explicación teórica:* `lecturas_crudas * pendientes` arroja un `ValueError` porque las formas `(5, 7)` y `(5,)` no coinciden de derecha a izquierda. Para solucionarlo, se redimensionan los vectores de calibración a `(5, 1)` permitiendo el broadcasting automático sobre las columnas de cada sensor.

In [4]:
# Redimensionar a vectores columna de forma (5, 1)
pendientes_col = pendientes[:, np.newaxis]
interceptos_col = interceptos[:, np.newaxis]

# Calibración vectorizada de toda la red
calibradas = lecturas_crudas * pendientes_col + interceptos_col

# Calibración de la campaña de SBC-A (pendiente 1.2, intercepto -0.5)
campana_calibrada = lecturas_utiles * 1.2 - 0.5
promedio_calibrado_sbc_a = campana_calibrada.mean()

print("Forma de calibradas:", calibradas.shape)
print("Primera fila calibrada:", calibradas[0])
print("Campana calibrada SBC-A:", campana_calibrada)
print("Promedio calibrado SBC-A:", promedio_calibrado_sbc_a)

Forma de calibradas: (5, 7)
Primera fila calibrada: [11.5 16.9 25.9 36.1 33.1 21.1 10.9]
Campana calibrada SBC-A: [11.5 29.5 47.5  9.1 61.9]
Promedio calibrado SBC-A: 31.9


### Ejercicio 5 — Resumir la semana con estadística por eje
Se calculan los agregados estadísticos globales, por sensor (`axis=1`), por día (`axis=0`), el máximo por sensor y el promedio de las celdas que superan el umbral de `35.4`, empleando las funciones nativas de NumPy y redondeando los resultados.

In [5]:
promedio_general = calibradas.mean()
promedio_por_sensor = calibradas.mean(axis=1)
promedio_por_dia = calibradas.mean(axis=0)
maximo_por_sensor = calibradas.max(axis=1)

mascara_sobre_limite = calibradas > 35.4
media_sobre_limite = calibradas[mascara_sobre_limite].mean()

print("Promedio general:", promedio_general)
print("Promedio por sensor:", np.round(promedio_por_sensor, 2))
print("Promedio por día:", np.round(promedio_por_dia, 2))
print("Máximo por sensor:", maximo_por_sensor)
print("Media sobre el límite (35.4):", media_sobre_limite)

Promedio general: 24.10857142857143
Promedio por sensor: [22.21 26.07 17.64 31.79 22.83]
Promedio por día: [10.12 16.14 25.62 35.92 42.96 27.94 10.06]
Máximo por sensor: [36.1 44.2 34.3 52.5 57. ]
Media sobre el límite (35.4): 43.275000000000006


### Ejercicio 6 — Funciones universales y reemplazo con máscara
Se hace uso de funciones universales como `np.maximum` y `np.abs` para cuantificar excesos y desvíos frente al límite de `35.4`, y se genera una copia de seguridad con `calibradas.copy()` para aplicar un reemplazo condicional sin alterar la matriz original.

In [6]:
excedente = np.maximum(calibradas - 35.4, 0)
excedente_por_sensor = np.round(excedente.sum(axis=1), 1)

desvio_sensor_b = np.round(np.abs(calibradas[1] - 35.4), 1)

en_alerta = calibradas.copy()
en_alerta[en_alerta <= 35.4] = 0.0

celdas_en_alerta = int((en_alerta > 0).sum())

print("Excedente por sensor:", excedente_por_sensor)
print("Desvío sensor B:", desvio_sensor_b)
print("Celdas en alerta mayores que 0:", celdas_en_alerta)
print("¿Calibradas sigue intacta?:", calibradas[0, 3] == 36.1)

Excedente por sensor: [ 0.7 11.3  0.  27.3 23.7]
Desvío sensor B: [23.6 16.4  2.9  8.8  2.5 11.9 21.8]
Celdas en alerta mayores que 0: 8
¿Calibradas sigue intacta?: True


### Ejercicio 7 — Integración: el informe de la red
Se implementan máscaras booleanas estrictas sobre el arreglo `calibradas` para contabilizar las mediciones en cada una de las cuatro categorías oficiales de calidad del aire sin solapamientos, y se identifica dinámicamente el día pico de contaminación con base en los promedios diarios.

In [7]:
buenas = int((calibradas <= 12.0).sum())
moderadas = int(((calibradas > 12.0) & (calibradas <= 35.4)).sum())
daninas_sensibles = int(((calibradas > 35.4) & (calibradas <= 55.4)).sum())
daninas_salud = int((calibradas > 55.4).sum())

# Identificación del día pico
mascara_dia_pico = (promedio_por_dia == promedio_por_dia.max())
dia_pico = int(dias[mascara_dia_pico][0])
valor_dia_pico = round(float(promedio_por_dia.max()), 1)

print(f"Lecturas por categoría: Buena={buenas}, Moderada={moderadas}, Dañina para grupos sensibles={daninas_sensibles}, Dañina para la salud={daninas_salud}")
print(f"Día más contaminado: {dia_pico} con {valor_dia_pico} ug/m3")

Lecturas por categoría: Buena=9, Moderada=18, Dañina para grupos sensibles=7, Dañina para la salud=1
Día más contaminado: 5 con 43.0 ug/m3


In [8]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert lecturas_crudas.shape == (5, 7) and lecturas_crudas.ndim == 2, "revise lecturas_crudas del Ejercicio 1"
assert forma == (5, 7) and dimensiones == 2 and total == 35 and tipo == np.float64, "revise los atributos del Ejercicio 1"
assert np.array_equal(dias, np.arange(1, 8)), "dias debe ir del 1 al 7 (el 7 incluido)"
assert np.allclose(pendientes, [1.2, 0.9, 1.1, 1.0, 1.3]) and np.allclose(interceptos, [-0.5, 1.0, -2.0, 0.5, -1.5]), "revise pendientes e interceptos"
assert lectura_c_dia4 == 27.0 and ultimo_primer == 5.0, "revise la indexación del Ejercicio 2"
assert np.array_equal(dia5, [28.0, 41.0, 33.0, 52.0, 45.0]) and semana_b.shape == (7,), "revise dia5 y semana_b del Ejercicio 2"
assert bloque.shape == (3, 4) and bloque[0, 0] == 35.0, "revise el bloque del Ejercicio 2"
assert int(faltantes.sum()) == 2 and lecturas_utiles.shape == (5,), "revise las máscaras del Ejercicio 3"
assert abs(promedio_utiles - 27.0) < 1e-9, "revise promedio_utiles del Ejercicio 3"
assert calibradas.shape == (5, 7), "calibradas debe tener forma (5, 7)"
assert np.allclose(calibradas[0], [11.5, 16.9, 25.9, 36.1, 33.1, 21.1, 10.9]), "revise la calibración del Ejercicio 4"
assert lecturas_crudas[0, 0] == 10.0, "lecturas_crudas no debe modificarse"
assert abs(promedio_calibrado_sbc_a - 31.9) < 1e-9, "revise promedio_calibrado_sbc_a del Ejercicio 4"
assert abs(promedio_general - 24.1086) < 1e-4, "revise promedio_general del Ejercicio 5"
assert np.allclose(np.round(promedio_por_sensor, 2), [22.21, 26.07, 17.64, 31.79, 22.83]), "revise promedio_por_sensor (axis=1)"
assert np.allclose(np.round(promedio_por_dia, 2), [10.12, 16.14, 25.62, 35.92, 42.96, 27.94, 10.06]), "revise promedio_por_dia (axis=0)"
assert np.allclose(maximo_por_sensor, [36.1, 44.2, 34.3, 52.5, 57.0]), "revise maximo_por_sensor"
assert abs(media_sobre_limite - 43.275) < 1e-9, "revise media_sobre_limite"
assert np.allclose(excedente_por_sensor, [0.7, 11.3, 0.0, 27.3, 23.7]), "revise excedente_por_sensor del Ejercicio 6"
assert np.allclose(desvio_sensor_b, [23.6, 16.4, 2.9, 8.8, 2.5, 11.9, 21.8]), "revise desvio_sensor_b"
assert en_alerta.shape == (5, 7) and int((en_alerta > 0).sum()) == 8, "revise en_alerta"
assert abs(calibradas[0, 3] - 36.1) < 1e-9, "calibradas no debe modificarse al crear en_alerta"
assert (buenas, moderadas, daninas_sensibles, daninas_salud) == (9, 18, 7, 1), "revise los conteos del Ejercicio 7"
assert buenas + moderadas + daninas_sensibles + daninas_salud == calibradas.size, "los cuatro conteos deben sumar 35"
assert dia_pico == 5, "revise dia_pico del Ejercicio 7"
print("Verificación completada sin errores.")

Verificación completada sin errores.
